# Notebook 06 — 3D Brain Atrophy Visualisation

**What this notebook does:**
1. Generates a procedural MNI-inspired brain surface (no MRI files needed)
2. Maps real hippocampus volume data from `ADNIMERGE.csv` onto the surface as an atrophy heatmap
3. Renders **side-by-side interactive 3D brains**: baseline vs 24-month for demo patient RID 750
4. Saves a static PNG to `results/visualizations/brain_atrophy_3d.png`

**Prerequisites:** Run `02_lstm_model.ipynb` first to produce `demo_data.json`.  
**No MRI scans required** — the brain shape comes from a procedural ellipsoid with sulcal folds.

**Atrophy regions modelled:**
- Hippocampus (memory — first to shrink in AD)
- Entorhinal cortex (adjacent to hippocampus)
- Ventricles (expand as brain shrinks — shown as surface proxy)
- Frontal & parietal cortex (later-stage atrophy)

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Output paths
OUT_DIR = Path('results/visualizations')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_HTML = OUT_DIR / 'brain_atrophy_3d.html'
OUT_PNG  = OUT_DIR / 'brain_atrophy_3d.png'

print('Imports OK')

In [ ]:
# ── Cell 2: Load ADNIMERGE and extract atrophy data ───────────────────────────
df = pd.read_csv('data/raw/ADNIMERGE.csv')

DEMO_RIDS = [750, 667, 1282, 50, 128]

# Focus on RID 750 for the side-by-side (most complete bl + m24 data)
RID = 750

pat = df[df['RID'] == RID].sort_values('visit_num').reset_index(drop=True)
print(pat[['VISCODE', 'visit_num', 'MMSE', 'Hippocampus', 'DX', 'APOE4']].to_string())

# Baseline (visit 0)
bl  = pat[pat['VISCODE'] == 'bl'].iloc[0]
# 24-month (m24)
m24 = pat[pat['VISCODE'] == 'm24'].iloc[0]

hippo_bl  = float(bl['Hippocampus'])
hippo_m24 = float(m24['Hippocampus'])
mmse_bl   = float(bl['MMSE'])
mmse_m24  = float(m24['MMSE'])
apoe4     = int(bl['APOE4'])

hippo_pct_loss = (hippo_bl - hippo_m24) / hippo_bl * 100

print(f'\nRID {RID} | APOE4={apoe4}')
print(f'Hippocampus : {hippo_bl:.0f} mm³ → {hippo_m24:.0f} mm³  ({hippo_pct_loss:.1f}% loss)')
print(f'MMSE        : {mmse_bl:.0f} → {mmse_m24:.0f}')
print(f'Diagnosis   : {bl["DX"]} → {m24["DX"]}')

In [ ]:
# ── Cell 3: Load REAL brain surface from nilearn (fsaverage5) ────────────────
#
# nilearn ships actual MRI-derived brain meshes.
# fsaverage5 has ~10k vertices — fast to render, looks anatomically correct.
# We use the LEFT hemisphere pial surface (the outer cortical surface).

from nilearn import datasets, surface
import numpy as np

# Download/load fsaverage5 (cached after first run, ~5 MB)
fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')

# Load LEFT hemisphere pial surface (outer cortex)
coords_L, faces_L = surface.load_surf_mesh(fsaverage.pial_left)
coords_R, faces_R = surface.load_surf_mesh(fsaverage.pial_right)

# Stack both hemispheres, offset right hemisphere vertex indices
n_L = len(coords_L)
coords = np.vstack([coords_L, coords_R])
faces  = np.vstack([faces_L, faces_R + n_L])

# Extract x, y, z (RAS coordinates in mm)
bx, by, bz = coords[:, 0], coords[:, 1], coords[:, 2]

print(f'Vertices: {len(coords):,}   Faces: {len(faces):,}')
print(f'x (L-R):   [{bx.min():.0f}, {bx.max():.0f}] = {bx.max()-bx.min():.0f}mm')
print(f'y (A-P):   [{by.min():.0f}, {by.max():.0f}] = {by.max()-by.min():.0f}mm')
print(f'z (height):[{bz.min():.0f}, {bz.max():.0f}] = {bz.max()-bz.min():.0f}mm')
print('Real anatomical brain mesh loaded ✓')


In [ ]:
# ── Cell 4: Region weight maps on real brain vertices ────────────────────────
#
# MNI152 approximate region centres (RAS mm):
#   Hippocampus bilateral:  (±28, -20, -18)
#   Entorhinal:             (±25, -10, -30)
#   Amygdala:               (±24, -4,  -20)
#   Precuneus/parietal:     (  0, -65,  40)
#   Frontal (DLPFC):        (±35,  45,  25)
#   Posterior cingulate:    (  0, -50,  25)

import numpy as np

def gaussian_blob(vx, vy, vz, cx, cy, cz, sx, sy, sz):
    return np.exp(-(
        ((vx - cx)**2) / (2*sx**2) +
        ((vy - cy)**2) / (2*sy**2) +
        ((vz - cz)**2) / (2*sz**2)
    ))

def compute_region_weights(vx, vy, vz):
    # Bilateral = left + right hemisphere blobs
    hippo = (gaussian_blob(vx,vy,vz,  28,-20,-18, 12,12,10) +
             gaussian_blob(vx,vy,vz, -28,-20,-18, 12,12,10))

    entorh = (gaussian_blob(vx,vy,vz,  25,-10,-30, 10,10, 9) +
              gaussian_blob(vx,vy,vz, -25,-10,-30, 10,10, 9))

    amyg   = (gaussian_blob(vx,vy,vz,  24, -4,-20,  9, 9, 9) +
              gaussian_blob(vx,vy,vz, -24, -4,-20,  9, 9, 9))

    parietal  = gaussian_blob(vx,vy,vz,   0,-65, 40, 30,18,18)

    frontal   = (gaussian_blob(vx,vy,vz,  35, 45, 25, 18,18,18) +
                 gaussian_blob(vx,vy,vz, -35, 45, 25, 18,18,18))

    cingulate = gaussian_blob(vx,vy,vz,   0,-50, 25, 10,18,15)

    return dict(hippocampus=hippo, entorhinal=entorh, amygdala=amyg,
                parietal=parietal, frontal=frontal, cingulate=cingulate)

weights = compute_region_weights(bx, by, bz)

print('Region weights on real mesh:')
for k, v in weights.items():
    print(f'  {k:15s}: max={v.max():.3f}  (n_affected={( v>0.1).sum():,})')


In [ ]:
# ── Cell 5: Build atrophy intensity maps ─────────────────────────────────────
import numpy as np

def build_atrophy_map(hippo_vol, mmse_score, weights, hippo_ref=6000.0):
    hippo_severity = np.clip((hippo_ref - hippo_vol) / hippo_ref * 4.5, 0, 1)
    mmse_severity  = np.clip((30 - mmse_score) / 30, 0, 1)

    mtl      = (weights['hippocampus'] * 1.00
              + weights['entorhinal']  * 0.75
              + weights['amygdala']    * 0.55) * hippo_severity

    cortical = (weights['parietal']   * 0.55
              + weights['cingulate']  * 0.45
              + weights['frontal']    * 0.30) * mmse_severity

    atrophy = mtl + cortical
    atrophy = atrophy / (atrophy.max() + 1e-8)
    return atrophy

atr_bl  = build_atrophy_map(hippo_bl,  mmse_bl,  weights)
atr_m24 = build_atrophy_map(hippo_m24, mmse_m24, weights)

print(f'Baseline  atrophy — mean: {atr_bl.mean():.3f}, max: {atr_bl.max():.3f}')
print(f'24-month  atrophy — mean: {atr_m24.mean():.3f}, max: {atr_m24.max():.3f}')


In [ ]:
# ── Cell 6: Side-by-side 3D brain (Baseline | Latest visit) — all 5 patients ─
# Layout matches the reference design:
#   - Two subplots sharing the same 3D mesh, left=Baseline right=Latest
#   - Shared colorbar on the right (Healthy → Critical)
#   - Title block with patient stats
#   - Region legend at bottom
#   - No buttons/overlays — clean side-by-side comparison

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from nilearn import datasets, surface

OUT_DIR = Path('results/visualizations')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load mesh once ───────────────────────────────────────────────────────────
fsaverage        = datasets.fetch_surf_fsaverage(mesh='fsaverage5')
coords_L, faces_L = surface.load_surf_mesh(fsaverage.pial_left)
coords_R, faces_R = surface.load_surf_mesh(fsaverage.pial_right)
sulc_L           = surface.load_surf_data(fsaverage.sulc_left)
sulc_R           = surface.load_surf_data(fsaverage.sulc_right)

n_L    = len(coords_L)
coords = np.vstack([coords_L, coords_R])
faces  = np.vstack([faces_L, faces_R + n_L])
sulc   = np.concatenate([sulc_L, sulc_R])
bx, by, bz = coords[:, 0], coords[:, 1], coords[:, 2]
i_tri, j_tri, k_tri = faces[:, 0], faces[:, 1], faces[:, 2]
print(f'Mesh: {len(coords):,} verts  {len(faces):,} faces')

sulc_norm = (sulc - sulc.min()) / (sulc.max() - sulc.min())

# ── Colorscale (matches image 2: blue→teal→yellow→orange→red) ───────────────
CSCALE = [
    [0.00, '#05071a'],   # Healthy  — deep navy
    [0.12, '#0d1f6e'],   # Early    — dark blue
    [0.28, '#1a5fa8'],   # Mild     — steel blue
    [0.45, '#2ab8b0'],   # Moderate — teal
    [0.62, '#d4d44a'],   # Moderate — yellow
    [0.78, '#e87a30'],   # Severe   — orange
    [0.90, '#d43030'],   # Severe   — red
    [1.00, '#7a0000'],   # Critical — dark red
]

# ── Gaussian blobs ───────────────────────────────────────────────────────────
def gaussian_blob(vx, vy, vz, cx, cy, cz, sx, sy, sz):
    return np.exp(-(((vx-cx)**2)/(2*sx**2)
                   +((vy-cy)**2)/(2*sy**2)
                   +((vz-cz)**2)/(2*sz**2)))

def compute_weights(vx, vy, vz):
    return dict(
        hippo    = (gaussian_blob(vx,vy,vz, 28,-20,-18,14,14,12)
                  + gaussian_blob(vx,vy,vz,-28,-20,-18,14,14,12)),
        entorh   = (gaussian_blob(vx,vy,vz, 25,-10,-30,12,12,10)
                  + gaussian_blob(vx,vy,vz,-25,-10,-30,12,12,10)),
        amyg     = (gaussian_blob(vx,vy,vz, 24, -4,-20,10,10,10)
                  + gaussian_blob(vx,vy,vz,-24, -4,-20,10,10,10)),
        parietal  = gaussian_blob(vx,vy,vz,  0,-65, 40,32,20,20),
        frontal  = (gaussian_blob(vx,vy,vz, 35, 45, 25,20,20,20)
                  + gaussian_blob(vx,vy,vz,-35, 45, 25,20,20,20)),
        cingulate = gaussian_blob(vx,vy,vz,  0,-50, 25,12,20,17),
    )

weights = compute_weights(bx, by, bz)

def build_atrophy(hippo_vol, mmse_score, w, hippo_ref=6000.0):
    hs  = np.clip((hippo_ref - hippo_vol) / hippo_ref * 5.5, 0, 1)
    ms  = np.clip((30 - mmse_score) / 30 * 1.4, 0, 1)
    mtl = (w['hippo']*1.0 + w['entorh']*0.8 + w['amyg']*0.65) * hs
    ctx = (w['parietal']*0.7 + w['cingulate']*0.6 + w['frontal']*0.45) * ms
    a   = mtl + ctx
    return a / (a.max() + 1e-8)

def combined_intensity(atrophy, sulc_norm):
    """Blend sulcal folds (structural detail) with atrophy hot-signal."""
    base = sulc_norm * 0.28             # sulcal depth visible in blue range
    hot  = np.where(atrophy > 0.15,
                    0.28 + atrophy * 0.72,   # push into orange/red range
                    atrophy * 0.18)          # barely visible for healthy tissue
    intensity = np.maximum(base, hot)
    return intensity / (intensity.max() + 1e-8)

def make_mesh(intensity, showscale=False, colorbar_x=1.02):
    cb = dict(
        title=dict(text='Atrophy<br>severity', font=dict(color='#ccc', size=11)),
        tickvals=[0.02, 0.20, 0.45, 0.65, 0.82, 0.97],
        ticktext=['Healthy', 'Early', 'Mild', 'Moderate', 'Severe', 'Critical'],
        tickfont=dict(color='#ccc', size=10),
        len=0.65, thickness=14,
        x=colorbar_x, xanchor='left',
        bgcolor='rgba(0,0,0,0)', outlinewidth=0,
    ) if showscale else {}
    return go.Mesh3d(
        x=bx, y=by, z=bz,
        i=i_tri, j=j_tri, k=k_tri,
        intensity=intensity,
        colorscale=CSCALE,
        cmin=0, cmax=1,
        showscale=showscale,
        colorbar=cb,
        opacity=0.97,
        flatshading=False,
        lighting=dict(ambient=0.35, diffuse=0.75, specular=0.2,
                      roughness=0.65, fresnel=0.25),
        lightposition=dict(x=120, y=200, z=350),
        hoverinfo='skip',
    )

# ── Patient data ─────────────────────────────────────────────────────────────
from pathlib import Path as _P
_demo_path = _P('results/metrics/demo_data.json')

PATIENT_DATA = {
    750:  {'hippo_bl': 5256.0, 'hippo_end': 4836.0, 'mmse_bl': 27.0, 'mmse_end': 8.0,  'apoe4': 1, 'dx': 'Dementia'},
    667:  {'hippo_bl': 8116.0, 'hippo_end': 8038.0, 'mmse_bl': 28.0, 'mmse_end': 20.0, 'apoe4': 1, 'dx': 'Dementia'},
    1282: {'hippo_bl': 4601.0, 'hippo_end': 4143.0, 'mmse_bl': 26.0, 'mmse_end': 23.0, 'apoe4': 0, 'dx': 'Dementia'},
    50:   {'hippo_bl': 4831.0, 'hippo_end': 4556.0, 'mmse_bl': 25.0, 'mmse_end': 23.0, 'apoe4': 0, 'dx': 'Dementia'},
    128:  {'hippo_bl': 5703.0, 'hippo_end': 4985.0, 'mmse_bl': 29.0, 'mmse_end': 26.0, 'apoe4': 2, 'dx': 'Dementia'},
}

if _demo_path.is_file():
    import json as _j
    _demo = _j.loads(_demo_path.read_text())
    for rid_str, d in _demo.items():
        rid = int(rid_str)
        if rid in PATIENT_DATA and d.get('observed_hippo') and d.get('observed_mmse'):
            PATIENT_DATA[rid].update({
                'hippo_bl':  d['observed_hippo'][0],
                'hippo_end': d['observed_hippo'][-1],
                'mmse_bl':   d['observed_mmse'][0],
                'mmse_end':  d['observed_mmse'][-1],
                'apoe4':     d.get('apoe4', 0),
                'dx':        d.get('dx_last', 'Unknown'),
            })
    print('Patient data loaded from demo_data.json ✓')
else:
    print('Using hardcoded patient data (run 02_lstm_model.ipynb first)')

# ── Camera: slightly elevated front-left view ─────────────────────────────────
CAMERA = dict(
    eye=dict(x=1.55, y=-0.15, z=0.35),
    center=dict(x=0, y=0, z=-0.05),
    up=dict(x=0, y=0, z=1),
)

SCENE = dict(
    bgcolor='#050810',
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    zaxis=dict(visible=False),
    aspectmode='data',
    camera=CAMERA,
)

BG = '#060910'
FG = '#e8e8e8'

# ── Render HTML per patient ───────────────────────────────────────────────────
generated = []

for rid, p in PATIENT_DATA.items():
    h_bl  = p['hippo_bl'];  h_end = p['hippo_end']
    m_bl  = p['mmse_bl'];   m_end = p['mmse_end']
    apoe4 = int(p['apoe4'])
    dx    = p['dx']
    pct   = (h_bl - h_end) / h_bl * 100

    atr_bl  = build_atrophy(h_bl,  m_bl,  weights)
    atr_end = build_atrophy(h_end, m_end, weights)
    int_bl  = combined_intensity(atr_bl,  sulc_norm)
    int_end = combined_intensity(atr_end, sulc_norm)

    # ── Side-by-side subplots ─────────────────────────────────────────────────
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'mesh3d'}, {'type': 'mesh3d'}]],
        column_widths=[0.5, 0.5],
        horizontal_spacing=0.02,
    )

    fig.add_trace(make_mesh(int_bl,  showscale=False), row=1, col=1)
    fig.add_trace(make_mesh(int_end, showscale=True, colorbar_x=1.01), row=1, col=2)

    # ── Annotation text positions (paper coords) ──────────────────────────────
    annotations = [
        # Main title
        dict(text=f'<b>3D Brain Atrophy — Patient RID {rid}</b>',
             xref='paper', yref='paper', x=0.5, y=1.13,
             xanchor='center', yanchor='top',
             font=dict(size=17, color=FG, family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Subtitle
        dict(text=(
                 f'APOEε4={apoe4}  ·  '
                 f'Hippocampus loss <b>{pct:.1f}%</b>  ·  '
                 f'MMSE {m_bl:.0f} → {m_end:.0f}/30'
             ),
             xref='paper', yref='paper', x=0.5, y=1.065,
             xanchor='center', yanchor='top',
             font=dict(size=12, color='#aaa', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Baseline column header
        dict(text='<b style="color:#5bc4f5">Baseline</b>',
             xref='paper', yref='paper', x=0.235, y=1.01,
             xanchor='center', yanchor='top',
             font=dict(size=14, color='#5bc4f5', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Baseline sub-header
        dict(text=f'MMSE <b>{m_bl:.0f}</b>/30  ·  Hippocampus <b>{h_bl:,.0f}</b> mm³',
             xref='paper', yref='paper', x=0.235, y=0.975,
             xanchor='center', yanchor='top',
             font=dict(size=11, color='#aaa', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Latest column header
        dict(text='<b style="color:#ef6a6a">Latest visit</b>',
             xref='paper', yref='paper', x=0.755, y=1.01,
             xanchor='center', yanchor='top',
             font=dict(size=14, color='#ef6a6a', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Latest sub-header
        dict(text=f'MMSE <b>{m_end:.0f}</b>/30  ·  Hippocampus <b>{h_end:,.0f}</b> mm³',
             xref='paper', yref='paper', x=0.755, y=0.975,
             xanchor='center', yanchor='top',
             font=dict(size=11, color='#aaa', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Region legend — bottom centre
        dict(text=(
                 '<span style="color:#ef6a6a">●</span> '
                 'Hippocampus · Entorhinal · Amygdala '
                 '— medial temporal <i>(first affected)</i>'
             ),
             xref='paper', yref='paper', x=0.5, y=-0.04,
             xanchor='center', yanchor='top',
             font=dict(size=11, color='#ccc', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        dict(text=(
                 '<span style="color:#d4d44a">●</span> '
                 'Parietal · Cingulate · Frontal '
                 '— later-stage cortical spread'
             ),
             xref='paper', yref='paper', x=0.5, y=-0.09,
             xanchor='center', yanchor='top',
             font=dict(size=11, color='#ccc', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
        # Drag hint
        dict(text='✦ Drag to rotate  ·  Scroll to zoom  ·  Double-click to reset',
             xref='paper', yref='paper', x=0.5, y=-0.14,
             xanchor='center', yanchor='top',
             font=dict(size=10, color='#666', family='DM Sans, Inter, sans-serif'),
             showarrow=False),
    ]

    fig.update_layout(
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        height=560,
        margin=dict(l=0, r=80, t=95, b=90),
        font=dict(family='DM Sans, Inter, sans-serif', color=FG),
        annotations=annotations,
        scene=dict(**SCENE),
        scene2=dict(**SCENE),
        showlegend=False,
    )

    out_path = OUT_DIR / f'brain_atrophy_3d_{rid}.html'
    fig.write_html(
        str(out_path),
        config=dict(displayModeBar=False, responsive=True),
        include_plotlyjs='cdn',
        full_html=True,
    )
    generated.append(out_path)
    print(f'  ✓ RID {rid}: {h_bl:.0f}→{h_end:.0f} mm³  ({pct:.1f}% loss)  MMSE {m_bl:.0f}→{m_end:.0f}')

# ── Legacy alias ─────────────────────────────────────────────────────────────
import shutil
shutil.copy(OUT_DIR / 'brain_atrophy_3d_750.html',
            OUT_DIR / 'brain_atrophy_3d.html')
print(f'\n✓ Legacy brain_atrophy_3d.html → alias of RID 750')
print(f'Generated {len(generated)} files in {OUT_DIR}/')


In [ ]:
# ── Cell 7: Save static PNG (kaleido fix) ────────────────────────────────────
# If you get a kaleido error, run:  pip install -U kaleido
# Then RESTART the kernel and re-run from Cell 1.

import subprocess, sys

def ensure_kaleido():
    try:
        import kaleido
        return True
    except ImportError:
        print('kaleido not found — installing...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'kaleido', '-q'])
        print('kaleido installed. Restart kernel if this is the first install.')
        return False

ensure_kaleido()

try:
    fig.write_image(str(OUT_PNG), width=1100, height=560, scale=2)
    print(f'✓ Saved static PNG  → {OUT_PNG}')
    print(f'  This PNG will auto-appear in the Dash gallery.')
except Exception as e:
    print(f'PNG export skipped: {e}')
    print(f'  Workaround: open {OUT_HTML} in Chrome and use File → Print → Save as PDF,')
    print(f'  or take a screenshot and save as brain_atrophy_3d.png in results/visualizations/')


In [ ]:
# ── Cell 8: Summary bar — hippocampus % loss for all 5 demo patients ────────
import plotly.graph_objects as go

summary_rows = []
for rid, p in PATIENT_DATA.items():
    h_bl  = p['hippo_bl']
    h_end = p['hippo_end']
    m_bl  = p['mmse_bl']
    m_end = p['mmse_end']
    pct   = (h_bl - h_end) / h_bl * 100
    summary_rows.append({
        'RID':        rid,
        'APOE4':      int(p['apoe4']),
        'DX':         p['dx'],
        'Hippo BL':   h_bl,
        'Hippo End':  h_end,
        'Loss %':     round(pct, 2),
        'MMSE BL':    m_bl,
        'MMSE End':   m_end,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

colors = ['#ef4444' if r > 8 else '#f59e0b' if r > 4 else '#22c55e'
          for r in summary_df['Loss %']]

bar_fig = go.Figure([
    go.Bar(
        x=[f'RID {r}\n(APOE4={a})' for r, a in zip(summary_df['RID'], summary_df['APOE4'])],
        y=summary_df['Loss %'],
        marker_color=colors,
        text=[f"{v:.1f}%" for v in summary_df['Loss %']],
        textposition='outside',
        hovertemplate=(
            '<b>RID %{customdata[0]}</b><br>'
            'APOE4: %{customdata[1]}<br>'
            'Loss: %{y:.1f}%<br>'
            'Hippo: %{customdata[2]:,.0f} → %{customdata[3]:,.0f} mm³<br>'
            'MMSE:  %{customdata[4]:.0f} → %{customdata[5]:.0f}<extra></extra>'
        ),
        customdata=list(zip(
            summary_df['RID'], summary_df['APOE4'],
            summary_df['Hippo BL'], summary_df['Hippo End'],
            summary_df['MMSE BL'], summary_df['MMSE End'],
        )),
    )
])
bar_fig.update_layout(
    title='Hippocampus volume loss — all demo patients',
    yaxis_title='Volume loss (%)',
    yaxis=dict(range=[0, max(summary_df['Loss %']) * 1.3]),
    paper_bgcolor='rgba(245,245,248,1)',
    plot_bgcolor='rgba(245,245,248,1)',
    height=400,
    font=dict(family='DM Sans, Inter, sans-serif'),
)
bar_fig.show()
print('Cell 8 complete — all patients summarised')


In [ ]:
# ── Cell 9: Done ────────────────────────────────────────────────────────────
print('06_brain_visualization complete!')
print()
print('Files written to results/visualizations/:')
for rid in PATIENT_DATA.keys():
    p = OUT_DIR / f'brain_atrophy_3d_{rid}.html'
    status = '✓' if p.is_file() else '✗'
    print(f'  {status} brain_atrophy_3d_{rid}.html')
print(f'  ✓ brain_atrophy_3d.html  (legacy alias for RID 750)')
print()
print('Reload app.py — the brain section now shows a per-patient dropdown.')
